<!-- cell 0 -->
# Notebook 6 — Stage 4: Evaluation
**MAI 656 — Natural Language Processing | Canadian University Dubai | Spring 2026**

This notebook evaluates the fine-tuned `Qwen/Qwen2.5-VL-7B-Instruct` + LoRA adapter on the held-out eval set, reporting:
- **CER** — Character Error Rate (via `jiwer`)
- **WER** — Word Error Rate (via `jiwer`)
- **Exact match rate**
- **Valid JSON rate**
- **Average inference time per sample**

**Reference labels.** Ground-truth transcriptions in `data/eval/eval.jsonl` were generated with **gpt-5.4-mini** during dataset construction. All metrics here measure the fine-tuned model's agreement with those GPT-5.4 references — not against gold human transcription.

> ⚠️ **A GPU with ≥24 GB VRAM is required** (same machine as training, or another GPU instance).

**Input:**
- `data/eval/eval.jsonl` — 280 eval samples (full set)
- `models/lora_adapter/run-2/` — adapter saved by Notebook 5

**Output:** `logs/run-2/eval_results.json`. Each evaluation writes into its run's log folder; switch `RUN_NAME` in the config cell to evaluate a different adapter (e.g. `run-1`).

<!-- cell 1 -->
## 1. Check GPU

In [1]:
# Cell 2
!nvidia-smi

Sat Apr 18 02:04:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.16             Driver Version: 580.126.16     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A10G                    On  |   00000000:00:1E.0 Off |                    0 |
|  0%   32C    P8             40W /  300W |       0MiB /  23028MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

<!-- cell 3 -->
## 2. Install Dependencies

In [2]:
# Cell 4
import sys
!{sys.executable} -m pip install "transformers>=4.52.0" peft==0.14.0 "accelerate>=1.3.0" "bitsandbytes>=0.46.1" torchvision
!{sys.executable} -m pip install qwen-vl-utils jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 69.8 MB/s eta 0:00:0000:01


<!-- cell 5 -->
## 3. Set Project Root

In [3]:
# Cell 6
from pathlib import Path

PROJECT_ROOT = Path('/home/ubuntu/nlp_project')

assert PROJECT_ROOT.exists(), f'Project root not found: {PROJECT_ROOT}'

print(f'Project root: {PROJECT_ROOT}')

Project root: /home/ubuntu/nlp_project


<!-- cell 7 -->
## 4. Configuration

In [14]:
# Cell 8
from pathlib import Path

BASE_MODEL = 'Qwen/Qwen2.5-VL-7B-Instruct'
MIN_PIXELS = 4   * 28 * 28   # must match training
MAX_PIXELS = 128 * 28 * 28   # must match training

RUN_NAME   = 'run-2'
# None = merged final adapter (= checkpoint-1680, end of epoch 3)
# 'checkpoint-1120' = best eval-loss checkpoint (end of epoch 2)
CHECKPOINT = 'checkpoint-1120'

ADAPTER_ROOT = Path(PROJECT_ROOT) / 'models' / 'lora_adapter' / RUN_NAME
ADAPTER_DIR  = ADAPTER_ROOT / CHECKPOINT if CHECKPOINT else ADAPTER_ROOT
EVAL_DATA    = Path(PROJECT_ROOT) / 'data' / 'eval' / 'eval.jsonl'
LOG_DIR      = Path(PROJECT_ROOT) / 'logs' / RUN_NAME

LOG_DIR.mkdir(parents=True, exist_ok=True)
assert ADAPTER_DIR.exists(), f'Adapter not found: {ADAPTER_DIR}'

print(f'Run:     {RUN_NAME}')
print(f'Ckpt:    {CHECKPOINT or "merged"}')
print(f'Adapter: {ADAPTER_DIR}')
print(f'Eval:    {EVAL_DATA}')
print(f'Log dir: {LOG_DIR}')

Run:     run-2
Ckpt:    checkpoint-1120
Adapter: /home/ubuntu/nlp_project/models/lora_adapter/run-2/checkpoint-1120
Eval:    /home/ubuntu/nlp_project/data/eval/eval.jsonl
Log dir: /home/ubuntu/nlp_project/logs/run-2


<!-- cell 9 -->
## 5. Load Base Model + LoRA Adapter

In [15]:
# Cell 10
import gc
import torch
from huggingface_hub import snapshot_download
from huggingface_hub.utils import LocalEntryNotFoundError
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)
from peft import PeftModel

HF_CACHE_DIR = None  # None = default ~/.cache/huggingface/hub/

# ---------- Slow path: load base model + processor once per kernel ----------
if 'base_model' not in globals() or 'processor' not in globals():
    print(f'Verifying cache for: {BASE_MODEL}')
    try:
        snapshot_path = snapshot_download(
            repo_id=BASE_MODEL, local_files_only=True, cache_dir=HF_CACHE_DIR,
        )
        print(f'  Cache OK — {snapshot_path}')
    except LocalEntryNotFoundError:
        print(f'  Cache MISS — downloading {BASE_MODEL} (~16 GB)...')
        snapshot_path = snapshot_download(repo_id=BASE_MODEL, cache_dir=HF_CACHE_DIR)
        print(f'  Download complete.')

    print('Loading base model into GPU memory...')
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        snapshot_path,
        quantization_config=quantization_config,
        torch_dtype=torch.bfloat16,
        device_map='auto',
        attn_implementation='sdpa',
        local_files_only=True,
    )

    processor = AutoProcessor.from_pretrained(
        snapshot_path,
        min_pixels=MIN_PIXELS,
        max_pixels=MAX_PIXELS,
        local_files_only=True,
    )
else:
    print('Base model + processor already in kernel — skipping 16 GB reload.')

# ---------- Fast path: (re)attach the LoRA adapter from ADAPTER_DIR ----------
# If a PeftModel already wraps the base, unload it cleanly before swapping
# checkpoints (prevents stacking a second 'default' adapter on top of the old one).
if 'model' in globals() and hasattr(model, 'unload'):
    print('Unloading previous adapter...')
    try:
        base_model = model.unload()
    except Exception as e:
        print(f'  (unload failed: {e} — reusing existing base_model)')
    del model
    gc.collect()
    torch.cuda.empty_cache()

print(f'Loading LoRA adapter: {ADAPTER_DIR}')
model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
model.eval()

print('Model and processor ready.')

Base model + processor already in kernel — skipping 16 GB reload.
Unloading previous adapter...
Loading LoRA adapter: /home/ubuntu/nlp_project/models/lora_adapter/run-2/checkpoint-1120
Model and processor ready.


<!-- cell 11 -->
## 6. Define Metrics and JSON Parser

CER and WER are computed with `jiwer`, which returns the standard `(Substitutions + Deletions + Insertions) / N_reference`. Values **can exceed 1.0** when the hypothesis is longer than the reference (insertions outnumber reference tokens) — we keep the raw ratios rather than clamping so catastrophic over-production stays visible in the average. `parse_output()` recovers structured JSON from the model's raw text — handling markdown fences, truncated braces, and unterminated strings — so downstream metrics work even when the model drifts from the strict `{"transcription":"..."}` template.

In [16]:
# Cell 12
import json
from jiwer import cer as _jiwer_cer, wer as _jiwer_wer

def parse_output(raw_text: str) -> dict:
    """Try multiple strategies to extract valid JSON."""
    text = raw_text.strip()

    # Strategy 1: Direct parse
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Strategy 2: Strip markdown code blocks
    if '```' in text:
        text = text.split('```json')[-1].split('```')[0].strip()
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            pass

    # Strategy 3: Find last closing brace
    for i in range(len(text) - 1, -1, -1):
        if text[i] == '}':
            try:
                return json.loads(text[:i+1])
            except json.JSONDecodeError:
                continue

    # Strategy 4: Auto-complete common truncations
    for suffix in ['"}',' "}', '"}', '"]}', '"}}']:
        try:
            return json.loads(text + suffix)
        except json.JSONDecodeError:
            continue

    return None


def char_error_rate(prediction: str, reference: str) -> float:
    if not reference:
        return 0.0 if not prediction else 1.0
    return float(_jiwer_cer(reference, prediction))


def word_error_rate(prediction: str, reference: str) -> float:
    if not reference.strip():
        return 0.0 if not prediction.strip() else 1.0
    return float(_jiwer_wer(reference, prediction))

print('Metric functions defined.')

Metric functions defined.


<!-- cell 13 -->
## 7. Run Evaluation

Split into three cells so each stage can be rerun independently:

- **7a.** Load eval samples & define the base64 image decoder
- **7b.** Configure generation (token suppression, KV cache, EOS/pad ids)
- **7c.** Generate predictions, score CER / WER / exact-match, and collect results

### 7a. Load Eval Samples & Image Decoder

In [20]:
# Cell 14
import time
import base64
from io import BytesIO
from PIL import Image

def extract_images(messages):
    """Decode base64 data URLs from OpenAI-style `image_url` content into PIL Images."""
    images = []
    for msg in messages:
        content = msg.get('content')
        if not isinstance(content, list):
            continue
        for item in content:
            t = item.get('type')
            if t == 'image_url':
                url = item['image_url']
                if isinstance(url, dict):
                    url = url.get('url', '')
                if url.startswith('data:'):
                    b64 = url.split(',', 1)[1]
                    images.append(Image.open(BytesIO(base64.b64decode(b64))).convert('RGB'))
                elif url:
                    images.append(Image.open(url).convert('RGB'))
            elif t == 'image':
                src = item.get('image')
                if isinstance(src, Image.Image):
                    images.append(src)
                elif isinstance(src, str) and src.startswith('data:'):
                    b64 = src.split(',', 1)[1]
                    images.append(Image.open(BytesIO(base64.b64decode(b64))).convert('RGB'))
                elif isinstance(src, str):
                    images.append(Image.open(src).convert('RGB'))
    return images

# Load eval samples
eval_samples = []
with open(EVAL_DATA) as f:
    for line in f:
        eval_samples.append(json.loads(line))
print(f'Evaluating {len(eval_samples)} samples...')

Evaluating 280 samples...


<!-- cell 15 -->
### 7b. Configure Generation

Build the bad-words list that blocks `<tool_call>` hallucinations, re-enable the KV cache (training turns it off for gradient checkpointing), and resolve `eos` / `pad` token ids once so the generation loop doesn't repeat the work.

In [21]:
# Cell 16
# Build token suppression list (prevents <tool_call> hallucinations)
suppress_tokens = ['<tool_call>', '<|tool_call|>', '<tool_response>', '<|tool_response|>']
bad_words_ids = []
for token in suppress_tokens:
    ids = processor.tokenizer.encode(token, add_special_tokens=False)
    if ids:
        bad_words_ids.append(ids)

# Re-enable KV cache for inference (training disabled it for gradient checkpointing)
model.config.use_cache = True

eos_token_id = processor.tokenizer.eos_token_id
pad_token_id = processor.tokenizer.pad_token_id or eos_token_id

print(f'Suppressed token sequences: {len(bad_words_ids)}')
print(f'eos_token_id={eos_token_id}  pad_token_id={pad_token_id}')

Suppressed token sequences: 4
eos_token_id=151645  pad_token_id=151643


<!-- cell 17 -->
### 7c. Generate & Score

Iterate the eval set: apply the chat template, force the `{"transcription":"` JSON prefix, generate with `do_sample=False`, parse the output, and record CER / WER / exact-match / valid-JSON / latency for each sample.

In [23]:
# Cell 18
results = []
for i, sample in enumerate(eval_samples):
    messages = sample['messages']
    expected = messages[-1]['content']      # ground truth
    input_messages = messages[:-1]          # system + user

    text = processor.apply_chat_template(
        input_messages, tokenize=False, add_generation_prompt=True
    )

    # JSON prefix forcing — steers the model to produce valid JSON immediately
    json_prefix = '{"transcription":"'
    text += json_prefix

    # Decode base64 images directly (eval data uses OpenAI-style image_url format)
    image_inputs = extract_images(input_messages)

    inputs = processor(
        text=[text],
        images=image_inputs if image_inputs else None,
        padding=True,
        return_tensors='pt',
    ).to(model.device)

    start_time = time.time()
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,          # transcriptions are short (~5–20 words)
            do_sample=False,
            use_cache=True,              # KV cache on → big latency win
            repetition_penalty=1.1,
            no_repeat_ngram_size=6,
            eos_token_id=eos_token_id,
            pad_token_id=pad_token_id,
            bad_words_ids=bad_words_ids if bad_words_ids else None,
            stop_strings=['"}'],         # halt the moment JSON closes → caps over-production
            tokenizer=processor.tokenizer,
        )
    inference_time = time.time() - start_time

    generated = processor.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    generated = json_prefix + generated  # re-attach prefix

    pred_parsed = parse_output(generated)
    ref_parsed  = parse_output(expected)

    pred_text = pred_parsed.get('transcription', '') if pred_parsed else ''
    ref_text  = ref_parsed.get('transcription',  '') if ref_parsed  else ''

    cer   = char_error_rate(pred_text, ref_text)
    wer   = word_error_rate(pred_text, ref_text)
    exact = pred_text.strip() == ref_text.strip()

    results.append({
        'cer': cer, 'wer': wer,
        'exact_match': exact,
        'valid_json': pred_parsed is not None,
        'inference_time': inference_time,
        'prediction': pred_text,
        'reference': ref_text,
    })

    print(f'[{i+1}/{len(eval_samples)}] CER={cer:.3f}  WER={wer:.3f}  Time={inference_time:.1f}s')

print('\nEvaluation complete.')

[1/280] CER=2.762  WER=3.000  Time=4.5s
[2/280] CER=0.818  WER=1.000  Time=4.6s
[3/280] CER=1.019  WER=1.000  Time=4.6s
[4/280] CER=0.838  WER=1.000  Time=4.5s
[5/280] CER=0.829  WER=1.000  Time=4.5s
[6/280] CER=1.867  WER=1.500  Time=4.5s
[7/280] CER=0.934  WER=1.000  Time=4.5s
[8/280] CER=0.897  WER=1.000  Time=4.5s
[9/280] CER=0.864  WER=1.000  Time=4.5s
[10/280] CER=0.964  WER=1.000  Time=4.5s
[11/280] CER=0.833  WER=1.000  Time=4.6s
[12/280] CER=1.178  WER=1.000  Time=4.5s
[13/280] CER=0.833  WER=1.000  Time=4.6s
[14/280] CER=1.170  WER=1.000  Time=4.6s
[15/280] CER=0.966  WER=1.000  Time=4.5s
[16/280] CER=7.333  WER=4.500  Time=4.5s
[17/280] CER=0.897  WER=1.000  Time=4.5s
[18/280] CER=1.541  WER=1.286  Time=4.5s
[19/280] CER=0.841  WER=1.000  Time=4.5s
[20/280] CER=4.133  WER=3.000  Time=4.5s
[21/280] CER=0.836  WER=1.000  Time=4.5s
[22/280] CER=1.222  WER=1.125  Time=4.5s
[23/280] CER=0.915  WER=1.000  Time=4.5s
[24/280] CER=1.057  WER=1.000  Time=4.5s
[25/280] CER=0.795  WER=1

In [11]:
# Cell 19
print(model.peft_config)          # shows the adapter path that was loaded
print(next(iter(model.peft_config.values())).base_model_name_or_path)

{'default': LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path='/home/ubuntu/.cache/huggingface/hub/models--Qwen--Qwen2.5-VL-7B-Instruct/snapshots/cc594898137f460bfe9f0759e9844b3ce807cfb5', revision=None, inference_mode=True, r=32, target_modules={'v_proj', 'q_proj', 'down_proj', 'o_proj', 'gate_proj', 'up_proj', 'k_proj'}, exclude_modules=None, lora_alpha=64, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', loftq_config={}, eva_config=None, use_dora=False, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False)}
/home/ubuntu/.cache/huggingface/hub/models--Qwen--Qwen2.5-VL-7B-Instruct/snapshots/cc594898137f460bfe9f0759e9844b3ce807cfb5


<!-- cell 20 -->
## 8. Summary & Save Results

In [24]:
# Cell 21
n = len(results)

avg_cer    = sum(r['cer']            for r in results) / n
avg_wer    = sum(r['wer']            for r in results) / n
exact_rate = sum(r['exact_match']    for r in results) / n * 100
json_rate  = sum(r['valid_json']     for r in results) / n * 100
avg_time   = sum(r['inference_time'] for r in results) / n

# Fall back gracefully if §4 wasn't re-run in this kernel session
checkpoint_name = globals().get('CHECKPOINT') or 'merged'
log_dir         = globals().get('LOG_DIR') or (Path(PROJECT_ROOT) / 'logs' / globals().get('RUN_NAME', 'run-2'))

summary = {
    'num_samples':        n,
    'checkpoint':         checkpoint_name,
    'avg_cer':            round(avg_cer,    4),
    'avg_wer':            round(avg_wer,    4),
    'exact_match_rate':   round(exact_rate, 2),
    'valid_json_rate':    round(json_rate,  2),
    'avg_inference_time': round(avg_time,   2),
}

print('=== Evaluation Summary ===')
for k, v in summary.items():
    print(f'  {k}: {v}')

# Tag filename with checkpoint so multiple evals don't overwrite each other
suffix = f'_{checkpoint_name}' if checkpoint_name != 'merged' else ''
output_path = Path(log_dir) / f'eval_results{suffix}.json'
output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f'\nResults saved to {output_path}')

=== Evaluation Summary ===
  num_samples: 280
  checkpoint: checkpoint-1120
  avg_cer: 1.2824
  avg_wer: 1.2767
  exact_match_rate: 0.0
  valid_json_rate: 100.0
  avg_inference_time: 4.52

Results saved to /home/ubuntu/nlp_project/logs/run-2/eval_results_checkpoint-1120.json


<!-- cell 22 -->
## 9. Inspect Worst Predictions

In [25]:
# Cell 23
# Show the 5 samples with the highest CER
worst = sorted(results, key=lambda r: r['cer'], reverse=True)[:5]

print('=== Top 5 Highest CER Samples ===')
for i, r in enumerate(worst):
    print(f'\n#{i+1}  CER={r["cer"]:.3f}  WER={r["wer"]:.3f}')
    print(f'  Reference:  {r["reference"]}')
    print(f'  Prediction: {r["prediction"]}')

=== Top 5 Highest CER Samples ===

#1  CER=13.200  WER=9.000
  Reference:  جميلة
  Prediction: الصينية، وهم يدعونها باللغة الصينية، ويعرفونها بـ "الصينية"، ويعرفونها

#2  CER=11.167  WER=9.000
  Reference:  تأشيرة
  Prediction: الصينية، وهم يدعونها باللغة الصينية، ويعرفونها بـ "الصينية"، ويعرفونها

#3  CER=9.000  WER=9.000
  Reference:  الصنيون
  Prediction: الصينية، وهم يدعونها باللغة الصينية، ويعرفونها بـ "الصينية"، ويعرفونها

#4  CER=8.250  WER=9.000
  Reference:  فنكتمكنه
  Prediction: الصينية، وهم يدعونها باللغة الصينية، ويعرفونها بـ "الصينية"، ويعرفونها

#5  CER=8.000  WER=4.500
  Reference:  ان يقبسه
  Prediction: الصينية، وهم يدعونها باللغة الصينية، ويعرفونها بـ "الصينية"، ويعرفونها


In [26]:
import statistics
buckets = {'short (≤10)': [], 'med (11-60)': [], 'long (>60)': []}
for r in results:
    n = len(r['reference'])
    key = 'short (≤10)' if n <= 10 else 'med (11-60)' if n <= 60 else 'long (>60)'
    buckets[key].append(r['cer'])
for k, v in buckets.items():
    print(f'{k:14s} n={len(v):3d}  mean={statistics.mean(v):.3f}  median={statistics.median(v):.3f}  p90={sorted(v)[int(0.9*len(v))-1]:.3f}')

short (≤10)    n=  8  mean=8.978  median=8.125  p90=11.167
med (11-60)    n=123  mean=1.330  median=1.060  p90=1.900
long (>60)     n=149  mean=0.829  median=0.829  p90=0.903


<!-- cell 24 -->
## Next Step (Optional)

If you want to serve the model as an API endpoint, continue to:

**Notebook 7 → `07_inference_server.ipynb`** — Launch a vLLM inference server